## Step 1: Feature Preparation
We select the required input features (`DMI (kg)`, `THI Range`, and `Genetic Group`) and the target variable (`Milk Yield (L/day/cow)`). We also include `Animal ID` temporarily to perform our grouped split. Then, we apply one-hot encoding to convert categorical variables (`THI Range` and `Genetic Group`) into numerical binary columns.

In [2]:
import pandas as pd

# Load dataset
df = pd.read_excel("milk yield and composition.xlsx")

# Select relevant columns including Animal ID for grouping
model_df = df[["Animal ID", "DMI (kg)", "THI Range", "Genetic Group", "Milk Yield (L/day/cow)"]].copy()

# One-hot encode categorical features
model_df = pd.get_dummies(model_df, columns=["THI Range", "Genetic Group"], dtype=int)

model_df.head()

,Animal ID,DMI (kg),Milk Yield (L/day/cow),THI Range_T0,THI Range_T1,THI Range_T2,Genetic Group_HF50,Genetic Group_HF62.5,Genetic Group_HF75,Genetic Group_HF87.5,Genetic Group_Local
0,102,6.91,2.05,1,0,0,0,0,0,0,1
1,102,6.61,1.14,1,0,0,0,0,0,0,1
2,102,7.31,1.74,1,0,0,0,0,0,0,1
3,102,6.75,1.96,1,0,0,0,0,0,0,1
4,102,6.42,2.24,1,0,0,0,0,0,0,1


## Step 2: 80/20 Grouped Split by Animal ID
Our dataset contains 750 records from 50 individual cows (15 records per cow). To prevent data leakage—ensuring records from the exact same cow do not appear in both training and testing sets—we use `GroupShuffleSplit` grouped by `Animal ID`.

In [4]:
from sklearn.model_selection import GroupShuffleSplit

# Separate features, target and group identifiers
X_full = model_df.drop(columns=["Milk Yield (L/day/cow)"])
y_full = model_df["Milk Yield (L/day/cow)"]
cow_groups = model_df["Animal ID"]

# Perform 80/20 grouped split
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(X_full, y_full, cow_groups))

# Drop Animal ID from feature matrices before training
X_train = X_full.drop(columns=["Animal ID"]).iloc[train_idx]
X_test = X_full.drop(columns=["Animal ID"]).iloc[test_idx]
y_train = y_full.iloc[train_idx]
y_test = y_full.iloc[test_idx]

print(f"Training cows: {df['Animal ID'].iloc[train_idx].nunique()} cows ({len(X_train)} records)")
print(f"Testing cows: {df['Animal ID'].iloc[test_idx].nunique()} cows ({len(X_test)} records)")

Training cows: 40 cows (600 records)
Testing cows: 10 cows (150 records)


## Step 3: Feature Scaling
We scale the continuous numerical feature (`DMI (kg)`) using `StandardScaler`. To prevent data leakage, the scaler is fitted strictly on the training set and then applied to transform both the training and testing sets.

In [5]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()

# Fit strictly on X_train, transform both
X_train_scaled[["DMI (kg)"]] = scaler.fit_transform(X_train[["DMI (kg)"]])
X_test_scaled[["DMI (kg)"]] = scaler.transform(X_test[["DMI (kg)"]])

X_train_scaled.head()

,DMI (kg),THI Range_T0,THI Range_T1,THI Range_T2,Genetic Group_HF50,Genetic Group_HF62.5,Genetic Group_HF75,Genetic Group_HF87.5,Genetic Group_Local
0,-1.133991,1,0,0,0,0,0,0,1
1,-1.250218,1,0,0,0,0,0,0,1
2,-0.979021,1,0,0,0,0,0,0,1
3,-1.195979,1,0,0,0,0,0,0,1
4,-1.323829,1,0,0,0,0,0,0,1


## Step 4: Train Multiple Linear Regression Model
We instantiate our first regression model (`LinearRegression`) and train (fit) it using the scaled training features and target variable.

In [11]:
from sklearn.linear_model import LinearRegression

# Initialize and fit the Multiple Linear Regression model
mlr_model = LinearRegression()
mlr_model.fit(X_train_scaled, y_train)

print("Multiple Linear Regression model successfully trained on training data!")

Multiple Linear Regression model successfully trained on training data!


## Step 5: Model Evaluation
We evaluate the trained model on the unseen test cows by generating predictions and calculating Mean Absolute Error (MAE), Root Mean Squared Error (RMSE), and Coefficient of Determination ($R^2$).

In [12]:
import numpy as np
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Generate predictions on test data
y_pred = mlr_model.predict(X_test_scaled)

# Calculate metrics
test_rmse = np.sqrt(mean_squared_error(y_test, y_pred))
test_mae = mean_absolute_error(y_test, y_pred)
test_r2 = r2_score(y_test, y_pred)

print(f"Test R² Score: {test_r2:.4f}")
print(f"Test RMSE: {test_rmse:.4f} Litres/day/cow")
print(f"Test MAE: {test_mae:.4f} Litres/day/cow")

Test R² Score: 0.8571
Test RMSE: 0.6071 Litres/day/cow
Test MAE: 0.4865 Litres/day/cow
